# JODI-Oil Secondary vs PPAC — India consumption comparison

This notebook does four things:

1. **Loads the consolidated JODI Secondary parquet** (refined products, ~15M rows, Jan 2002 onward) produced by `scripts/update_jodi.py`.
2. **Loads the PPAC PT-Consumption parquet** (India's product-wise petroleum consumption, fiscal-year sheets stitched together back to 1998-99) produced by `scripts/update_india_pt_consumption.py`.
3. **Compares them across the full overlapping window** at product and total levels — both at the latest snapshot and as time series — then quantifies where, when, and by how much the two sources diverge.
4. **Plots the PPAC-only series for the freshest months** that JODI hasn't published yet. PPAC publishes ~6 weeks ahead of JODI, so this is the timely view of what India's demand is doing right now.

## What the JODI file is

JODI publishes two databases: **Primary** (crude + NGLs + feedstocks) and **Secondary** (refined products). The file here is Secondary. JODI publishes with a ~2-month lag, so the freshest months in the parquet are typically two months behind the calendar.

## JODI schema

Every row is one observation keyed by **five dimensions**:

| Column | What it is |
|---|---|
| `REF_AREA` | ISO-2 country code (`IN` = India) |
| `TIME_PERIOD` | Month, `YYYY-MM` |
| `ENERGY_PRODUCT` | Refined product code |
| `FLOW_BREAKDOWN` | Balance item |
| `UNIT_MEASURE` | Same observation reported in 5 units |
| `OBS_VALUE` | The number — **stored as string** because flags like `-`, `x`, `c`, `..` are mixed in |
| `ASSESSMENT_CODE` | JODI's data-quality assessment flag — 1 = reasonable comparability, 2 = use with caution (consult metadata), 3 = not assessed, 4 = under verification. **Note**: this is about JODI's validation status, not who reported the number. Code 3 ≠ "JODI made it up". |

### ENERGY_PRODUCT codes
| Code | Meaning |
|---|---|
| `GASOLINE` | Motor gasoline |
| `GASDIES` | Gas/diesel oil (road diesel + heating gasoil) |
| `JETKERO` | Kerosene-type jet fuel. **In JODI's source taxonomy, JETKERO is a subset of KEROSENE** (the guide labels them "Kerosenes" with JETKERO as "of which: kerosene type jet fuel"). They appear as separate CSV rows but represent a hierarchy, not disjoint buckets — never sum them together as if they were independent. |
| `KEROSENE` | **Total kerosenes** (jet + non-jet). `JETKERO` is the "of which: jet" subset of this. So **non-jet kerosene = `KEROSENE − JETKERO`** — that's what maps to PPAC's SKO. |
| `LPG` | Liquefied petroleum gases |
| `NAPHTHA` | Naphtha |
| `RESFUEL` | Residual fuel oil |
| `ONONSPEC` | Other oil products — refinery gas, ethane, petcoke, lubricants, white spirit, bitumen, paraffin waxes, etc. |
| `TOTPRODS` | Total of the above |

### FLOW_BREAKDOWN codes
JODI's balance identity: **`RECEIPTS + REFGROUT + TOTIMPSB − TOTEXPSB − IPTRANSF − PTRANSF + STOCKCH + STATDIFF = TOTDEMO`**

| Code | Meaning |
|---|---|
| `RECEIPTS` | Receipts (supply-side flow alongside refinery output) |
| `REFGROUT` | Refinery gross output |
| `TOTIMPSB` | Total imports |
| `TOTEXPSB` | Total exports |
| `IPTRANSF` | Interproduct transfers |
| `PTRANSF` | Products transferred (to refinery feedstocks) |
| `STOCKCH` | Stock change (positive = build) |
| `CLOSTLV` | Closing stock **level** (not a flow) |
| `STATDIFF` | Statistical difference |
| `TOTDEMO` | **Total demand. Includes refinery fuel + international marine & aviation bunkers** — important caveat below |

### UNIT_MEASURE codes
| Code | Meaning |
|---|---|
| `KBD` | Thousand barrels per day — best for cross-country comparison |
| `KBBL` | Thousand barrels for the month |
| `KTONS` | **Thousand metric tons for the month — same unit PPAC uses** |
| `KL` | Thousand kilolitres for the month |
| `CONVBBL` | Conversion factor, bbl/ton (a constant, not an observation) |

In [25]:
"""Setup. Loads from the consolidated processed parquets so the comparison covers
the full overlap window (Jan 2002 onward), not just the in-year JODI CSV.

Plotly is used for charts (matches notebooks 04 and 05). Project root is resolved
the same way as those notebooks so this runs from either the repo root or notebooks/.
"""

from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)


def _resolve_project_root() -> Path:
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "scripts" / "update_jodi.py").exists():
            return candidate
        if (candidate / "country_oil_scraper" / "scripts" / "update_jodi.py").exists():
            return candidate / "country_oil_scraper"
    raise RuntimeError(f"Could not locate project root from cwd: {here}")


PROJECT_ROOT = _resolve_project_root()
JODI_PARQUET = PROJECT_ROOT / "data" / "processed" / "jodi" / "jodi_secondary.parquet"
PPAC_PARQUET = PROJECT_ROOT / "data" / "processed" / "india" / "india_pt_consumption.parquet"

print("Project root :", PROJECT_ROOT)
print("JODI parquet :", JODI_PARQUET, "exists =", JODI_PARQUET.exists())
print("PPAC parquet :", PPAC_PARQUET, "exists =", PPAC_PARQUET.exists())

Project root : c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\country_oil_scraper
JODI parquet : c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\country_oil_scraper\data\processed\jodi\jodi_secondary.parquet exists = True
PPAC parquet : c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\country_oil_scraper\data\processed\india\india_pt_consumption.parquet exists = True


## 1. Load JODI

The processed parquet is already cleaned — JODI's missing-flags (`-`, `x`, `c`, `..`) have been resolved to NaN and the value column is numeric. We just rename the lowercase processor columns back to the uppercase SDMX names the analysis cells below expect, and build a `Period[M]` time column from the parquet's `date` so monthly joins work cleanly.

In [26]:
_jodi_raw = pd.read_parquet(JODI_PARQUET)

# Rename to uppercase SDMX names so the rest of the notebook reads naturally.
# The pivot helpers in cell 5 and 7 below were written against this column convention.
jodi = _jodi_raw.rename(columns={
    'ref_area':        'REF_AREA',
    'energy_product':  'ENERGY_PRODUCT',
    'flow_breakdown':  'FLOW_BREAKDOWN',
    'unit_measure':    'UNIT_MEASURE',
    'obs_value':       'OBS_VALUE',
    'assessment_code': 'ASSESSMENT_CODE',
})

# Monthly Period — needed for the (month, product) join with PPAC further down.
jodi['TIME_PERIOD'] = pd.PeriodIndex(jodi['date'], freq='M')

# Re-derive ASSESSMENT here so the labels match the discussion in the takeaways section.
jodi['ASSESSMENT'] = jodi['ASSESSMENT_CODE'].map({
    1: 'comparable',     # assessment shows reasonable comparability
    2: 'use_caution',    # consult metadata
    3: 'not_assessed',   # cell exists but hasn't been validated yet (normal for fresh months)
    4: 'under_verif',    # under verification
}).astype('category')

print(f'Rows: {len(jodi):,}')
print(f'Countries: {jodi["REF_AREA"].nunique()}')
print(f'Time range: {jodi["TIME_PERIOD"].min()} -> {jodi["TIME_PERIOD"].max()}')
print(f'Assessment mix: {jodi["ASSESSMENT"].value_counts().to_dict()}')

Rows: 15,443,447
Countries: 118
Time range: 2002-01 -> 2026-04
Assessment mix: {'not_assessed': 12194447, 'comparable': 3019368, 'use_caution': 219385}


### Long → wide helper

The long SDMX layout is great for storage but painful for analysis. Pivot to `(REF_AREA, TIME_PERIOD, ENERGY_PRODUCT)` rows with one column per `FLOW_BREAKDOWN`. Pick **one unit** at a time — pivoting across all five blows up the column count.

In [27]:
def to_wide(df, unit='KTONS'):
    subset = df[df['UNIT_MEASURE'] == unit]
    wide = subset.pivot_table(
        index=['REF_AREA', 'TIME_PERIOD', 'ENERGY_PRODUCT'],
        columns='FLOW_BREAKDOWN',
        values='OBS_VALUE',
        aggfunc='first',
        observed=True,
    ).reset_index()
    wide.columns.name = None
    return wide

kt  = to_wide(jodi, 'KTONS')   # for comparing to PPAC
kbd = to_wide(jodi, 'KBD')     # for cross-country / time-series work
print('kt shape:', kt.shape, '| kbd shape:', kbd.shape)

kt shape: (210774, 13) | kbd shape: (207488, 11)


### Unit conversion helper

JODI stores the same observation in five units, but in many flow rows only KTONS is populated (the country's anchor unit). `CONVBBL` carries the conversion factor — **bbl per metric ton**, stored as `bbl/ton × 1000` in the raw column. So `CONVBBL = 7458` for India diesel means 7.458 bbl/ton.

Two non-obvious things to know before using this:

- **`CONVBBL` is per (country, month, product, FLOW).** For ~86% of country/product combos it's flow-invariant (same value for all flows), but for big refining/trading hubs (US, Singapore, Russia, China, Indonesia, Brunei, Malaysia, etc.) it varies because the product slate inside e.g. `GASDIES` is different for exports vs domestic demand. The helper joins at the flow level to handle both cases correctly.
- **Do not convert `TOTPRODS` rows directly.** The factor for an aggregate isn't flow-invariant in a clean way (each flow has a different product mix). The right move is to convert individual products and sum.

The formulas:

```
kbbl = ktons × bbl_per_ton           (because 1 kt = 1000 t, and 1000 t × X bbl/t = X kbbl)
kbd  = kbbl / days_in_month
kl   = ktons × kl_per_ton            (kl_per_ton inferred from any flow where both KL and KTONS are reported)
```

Some flows don't get a KBD value even when populated in other units — `CLOSTLV` is a level (point-in-time stock), and stock change is sometimes flagged that way too. Daily-rate units don't apply to stocks.

In [28]:
from calendar import monthrange
import warnings

def build_conversion_tables(jodi_df):
    """Build per-flow CONVBBL and kl/ton tables from a JODI long-format DataFrame.

    Excludes TOTPRODS — aggregate factors aren't flow-invariant; convert components then sum.
    """
    base = jodi_df[jodi_df['ENERGY_PRODUCT'] != 'TOTPRODS']

    # bbl/ton, joined at flow level (handles US/SG/RU etc. where slate differs by flow)
    conv_flow = (
        base[(base['UNIT_MEASURE']=='CONVBBL') & base['OBS_VALUE'].notna() & (base['OBS_VALUE']>0)]
        [['REF_AREA','TIME_PERIOD','ENERGY_PRODUCT','FLOW_BREAKDOWN','OBS_VALUE']]
        .rename(columns={'OBS_VALUE':'bbl_per_ton'})
        .copy()
    )
    conv_flow['bbl_per_ton'] = conv_flow['bbl_per_ton'] / 1000.0

    # kl/ton density inferred from KL/KTONS ratios on populated flows
    pairs = (
        base[base['UNIT_MEASURE'].isin(['KL','KTONS'])]
        .pivot_table(index=['REF_AREA','TIME_PERIOD','ENERGY_PRODUCT','FLOW_BREAKDOWN'],
                     columns='UNIT_MEASURE', values='OBS_VALUE', aggfunc='first', observed=True)
    )
    pairs['kl_per_ton'] = pairs['KL'] / pairs['KTONS']
    density_flow = pairs['kl_per_ton'].dropna()
    density_flow = density_flow[np.isfinite(density_flow) & (density_flow > 0)].reset_index()
    return conv_flow, density_flow


def convert_units(df, value_col='value', from_unit='KTONS', to_unit='KBD',
                  conv_flow=None, density_flow=None, warn_aggregates=True):
    """Convert JODI values between KTONS, KBBL, KBD, KL using JODI's own factors.

    df must have columns: REF_AREA, TIME_PERIOD (Period[M]), ENERGY_PRODUCT, FLOW_BREAKDOWN, and value_col.
    Returns df with an added '{value_col}_{to_unit}' column. NaN where factors aren't available.
    """
    if conv_flow is None or density_flow is None:
        raise ValueError('Pass conv_flow and density_flow from build_conversion_tables().')
    if warn_aggregates and (df['ENERGY_PRODUCT']=='TOTPRODS').any():
        warnings.warn(
            'TOTPRODS rows present: aggregate conversion is unreliable because product mix '
            'differs by flow. Convert individual products then sum.'
        )

    out = df.copy()
    if from_unit == to_unit:
        out[f'{value_col}_{to_unit}'] = out[value_col]
        return out

    keys = ['REF_AREA','TIME_PERIOD','ENERGY_PRODUCT','FLOW_BREAKDOWN']
    out = out.merge(conv_flow,    on=keys, how='left')
    out = out.merge(density_flow, on=keys, how='left')
    out['_days'] = out['TIME_PERIOD'].apply(
        lambda p: monthrange(p.year, p.month)[1] if pd.notna(p) else np.nan
    )

    # Go through KTONS as the common intermediate
    v = out[value_col]
    if   from_unit == 'KTONS': kt = v
    elif from_unit == 'KBBL':  kt = v / out['bbl_per_ton']
    elif from_unit == 'KBD':   kt = v * out['_days'] / out['bbl_per_ton']
    elif from_unit == 'KL':    kt = v / out['kl_per_ton']
    else: raise ValueError(f'Unsupported from_unit: {from_unit}')

    if   to_unit == 'KTONS': r = kt
    elif to_unit == 'KBBL':  r = kt * out['bbl_per_ton']
    elif to_unit == 'KBD':   r = kt * out['bbl_per_ton'] / out['_days']
    elif to_unit == 'KL':    r = kt * out['kl_per_ton']
    else: raise ValueError(f'Unsupported to_unit: {to_unit}')

    out[f'{value_col}_{to_unit}'] = r
    return out.drop(columns=['bbl_per_ton','kl_per_ton','_days'])


conv_flow, density_flow = build_conversion_tables(jodi)
print(f'CONVBBL rows: {len(conv_flow):,}  |  density rows: {len(density_flow):,}')

CONVBBL rows: 1,939,187  |  density rows: 840,182


**Quick test against the file's own reported KBD values** — the converter should reproduce them exactly.

In [29]:
# Pick India's diesel TOTDEMO Jan 2026 and convert from KTONS to KBD
demo = (
    jodi[(jodi['REF_AREA']=='IN') & (jodi['ENERGY_PRODUCT']=='GASDIES') &
         (jodi['FLOW_BREAKDOWN']=='TOTDEMO') & (jodi['UNIT_MEASURE']=='KTONS')]
    [['REF_AREA','TIME_PERIOD','ENERGY_PRODUCT','FLOW_BREAKDOWN','OBS_VALUE']]
    .rename(columns={'OBS_VALUE':'value'})
)
demo_converted = convert_units(demo, 'value', 'KTONS', 'KBD',
                               conv_flow=conv_flow, density_flow=density_flow)
print('Derived vs file:')
print(demo_converted.to_string(index=False))

expected = jodi[(jodi['REF_AREA']=='IN') & (jodi['ENERGY_PRODUCT']=='GASDIES') &
                (jodi['FLOW_BREAKDOWN']=='TOTDEMO') & (jodi['UNIT_MEASURE']=='KBD')]['OBS_VALUE']
print(f'File\'s reported KBD: {expected.tolist()}')

Derived vs file:
REF_AREA TIME_PERIOD ENERGY_PRODUCT FLOW_BREAKDOWN   value    value_KBD
      IN     2002-01        GASDIES        TOTDEMO  3166.0   761.678323
      IN     2002-02        GASDIES        TOTDEMO  2866.0   763.379571
      IN     2002-03        GASDIES        TOTDEMO  3405.0   819.177097
      IN     2002-04        GASDIES        TOTDEMO  3410.0      847.726
      IN     2002-05        GASDIES        TOTDEMO  3596.0      865.128
      IN     2002-06        GASDIES        TOTDEMO  3241.0     805.7126
      IN     2002-07        GASDIES        TOTDEMO  3255.0       783.09
      IN     2002-08        GASDIES        TOTDEMO  2912.0   700.570839
      IN     2002-09        GASDIES        TOTDEMO  2858.0     710.4988
      IN     2002-10        GASDIES        TOTDEMO  3231.0   777.316065
      IN     2002-11        GASDIES        TOTDEMO  3219.0     800.2434
      IN     2002-12        GASDIES        TOTDEMO  3525.0   848.046774
      IN     2003-01        GASDIES        TOTD

### Example — cross-country diesel ranking in kbd

This is the kind of view you can't get out of the wide-form `kbd` frame alone because not every country reports KBD directly — many only populate KTONS. Going KTONS → KBD via the converter fills those in.

In [30]:
latest_month = jodi['TIME_PERIOD'].max()

diesel_kt = (
    jodi[(jodi['ENERGY_PRODUCT']=='GASDIES') & (jodi['FLOW_BREAKDOWN']=='TOTDEMO') &
         (jodi['UNIT_MEASURE']=='KTONS') & (jodi['TIME_PERIOD']==latest_month)]
    [['REF_AREA','TIME_PERIOD','ENERGY_PRODUCT','FLOW_BREAKDOWN','OBS_VALUE']]
    .rename(columns={'OBS_VALUE':'demand_kt'})
)
diesel = convert_units(diesel_kt, 'demand_kt', 'KTONS', 'KBD',
                       conv_flow=conv_flow, density_flow=density_flow).dropna(subset=['demand_kt_KBD'])

print(f'Top 15 diesel consumers, {latest_month} (kbd):')
print(diesel.nlargest(15, 'demand_kt_KBD')[['REF_AREA','demand_kt','demand_kt_KBD']].round(2).to_string(index=False))

Top 15 diesel consumers, 2026-04 (kbd):
REF_AREA  demand_kt  demand_kt_KBD
      US    15489.0         3851.6
      CN    15091.0        3636.93
      FR     2743.0         682.09
      JP     2705.0         672.64
      ID     2544.0         627.44
      DE     2491.0         619.43
      CA     2427.0         603.51
      ES     2401.0         597.05
      SA    2416.22          596.0
      AU     2268.0         563.98
      TR     2114.0         525.68
      GB     2108.0         524.19
      IT     2006.0         498.83
      PL     1534.0         381.45
      TH     1504.0         365.87


## 2. Load PPAC

The processed parquet is already a tidy long-format frame (`scrapers/india_ppac.py` does the per-sheet header detection, melt, and fiscal-year stitching). All we do here is filter out the totals rows (we recompute them downstream so we know exactly what's in them), align on a `Period[M]` month, and rename the value column to `ppac_kt` so the join with JODI later is obvious.

In [31]:
VALID_PRODUCTS = ['LPG','Naphtha','MS','ATF','SKO','HSD','LDO','Lubricants & Greases',
                  'FO & LSHS','Bitumen','Petroleum coke','Others']

_ppac_raw = pd.read_parquet(PPAC_PARQUET)

# Drop totals rows — we sum the products ourselves in the totals-reconciliation cell.
# Filter to the PPAC product taxonomy we know how to map to JODI.
ppac = (
    _ppac_raw[~_ppac_raw['is_total_row']]
    .loc[lambda d: d['product'].isin(VALID_PRODUCTS)]
    .rename(columns={'value_000mt': 'ppac_kt'})
    [['date', 'product', 'ppac_kt', 'fiscal_year']]
    .copy()
)
ppac['month'] = pd.PeriodIndex(ppac['date'], freq='M')
ppac = ppac.drop(columns='date')

print(f'PPAC rows: {len(ppac):,} | products: {ppac["product"].nunique()} | months: {ppac["month"].nunique()}')
print(f'Time range: {ppac["month"].min()} -> {ppac["month"].max()}')
ppac.head()

PPAC rows: 4,068 | products: 12 | months: 339
Time range: 1998-04 -> 2026-06


,product,ppac_kt,fiscal_year,month
0,ATF,178.42775,1998-99,1998-04
2,Bitumen,229.66641,1998-99,1998-04
3,FO & LSHS,782.82218,1998-99,1998-04
4,HSD,3193.43384,1998-99,1998-04
5,LDO,89.10071,1998-99,1998-04


In [32]:
_ppac_raw

,date,calendar_year,calendar_month,month_name,product,product_canonical,category,metric_type,unit_measure,value_000mt,is_total_row,fiscal_year,fiscal_month,source_file,updated_at
0,1998-04-01,1998,4,APR,ATF,Jet Fuel,Kerosene,TOTDEMO,kt,178.42775,False,1998-99,1,1777985064_PT_Consumption_English.xls,2026-05-13 13:08:51.017022
1,1998-04-01,1998,4,APR,All Products total,None,None,TOTDEMO,kt,6674.72012,True,1998-99,1,1777985064_PT_Consumption_English.xls,2026-05-13 13:08:51.017022
2,1998-04-01,1998,4,APR,Bitumen,Bitumen,Heavy byproducts,TOTDEMO,kt,229.66641,False,1998-99,1,1777985064_PT_Consumption_English.xls,2026-05-13 13:08:51.017022
3,1998-04-01,1998,4,APR,FO & LSHS,Fuel Oil,Fuel Oil,TOTDEMO,kt,782.82218,False,1998-99,1,1777985064_PT_Consumption_English.xls,2026-05-13 13:08:51.017022
4,1998-04-01,1998,4,APR,HSD,Diesel,Distillates,TOTDEMO,kt,3193.43384,False,1998-99,1,1777985064_PT_Consumption_English.xls,2026-05-13 13:08:51.017022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4401,2026-06-01,2026,6,JUN,Naphtha,Naphtha,Naphtha,TOTDEMO,kt,562.00000,False,2026-27,3,2026_PT_Consumption_manual.xlsx,2026-07-06 14:59:45.001758
4402,2026-06-01,2026,6,JUN,Others,Others,Others,TOTDEMO,kt,533.00000,False,2026-27,3,2026_PT_Consumption_manual.xlsx,2026-07-06 14:59:45.001758
4403,2026-06-01,2026,6,JUN,Petroleum coke,Petcoke,Heavy byproducts,TOTDEMO,kt,1500.00000,False,2026-27,3,2026_PT_Consumption_manual.xlsx,2026-07-06 14:59:45.001758
4404,2026-06-01,2026,6,JUN,SKO,Kerosene,Kerosene,TOTDEMO,kt,33.00000,False,2026-27,3,2026_PT_Consumption_manual.xlsx,2026-07-06 14:59:45.001758


## 3. Product mapping (PPAC → JODI) — and friendly names

Six clean 1:1 matches, two definitional traps, and one bucket (`ONONSPEC`) that lumps several PPAC lines together. The "Display" column is what we use in chart titles and tables further down — the Indian-government acronyms (MS, HSD, ATF, SKO) are clear if you know the domain, but the friendly labels make the charts readable for everyone else.

| PPAC code | Display name | JODI | Match quality |
|---|---|---|---|
| `LPG`     | LPG       | `LPG`       | Clean |
| `MS`      | Gasoline  | `GASOLINE`  | Clean (note: PPAC's MS includes ethanol blending — that's a definitional choice, not an error) |
| `HSD`     | Diesel    | `GASDIES`   | Clean for India (HSD dominates the GASDIES bucket; LDO and heating gasoil are tiny here) |
| `ATF`     | Jet fuel  | `JETKERO`   | Clean |
| `Naphtha` | Naphtha   | `NAPHTHA`   | Clean |
| `FO & LSHS` | Fuel oil | `RESFUEL`  | Clean |
| `SKO`     | Kerosene  | `KEROSENE − JETKERO` | Clean once you understand the hierarchy. JODI's `KEROSENE` is total kerosenes including jet; `JETKERO` is the jet subset. Subtract to get non-jet kerosene, which is what PPAC reports as SKO. For India Jan 2026 this gives 865 − 828 = 37 kt vs PPAC's 38 kt — a 1 kt match. |
| `LDO`, `Lubricants & Greases`, `Bitumen`, `Petroleum coke`, `Others` | Light diesel oil, Lubricants, Bitumen, Petcoke, Others | part of `ONONSPEC` | JODI lumps all of these into `ONONSPEC` (the "other products" basket: refinery gas, petcoke, lubricants, white spirit, bitumen, paraffin waxes, etc.). To compare, sum the PPAC "others" basket against JODI's `ONONSPEC`. |

### The definitional gap that explains most of the residual

JODI's `TOTDEMO` is defined as **deliveries to inland market + refinery fuel + international marine and aviation bunkers**. PPAC publishes **domestic POL consumption** (inland sales by oil companies + private direct imports + SEZ DTA sales). So even when products map 1:1, you should expect JODI ≥ PPAC by roughly the bunkers + refinery-fuel volume — order of a few hundred kt/month for India.

In [33]:
# Clean 1:1 mappings
PPAC_TO_JODI_CLEAN = {
    'LPG': 'LPG',
    'MS': 'GASOLINE',
    'HSD': 'GASDIES',
    'ATF': 'JETKERO',
    'Naphtha': 'NAPHTHA',
    'FO & LSHS': 'RESFUEL',
    'SKO': 'KEROSENE_NONJET',   # synthesised below as KEROSENE - JETKERO
}
ppac['jodi_product'] = ppac['product'].map(PPAC_TO_JODI_CLEAN)

# Friendly display names. We keep the raw PPAC code as the canonical merge key
# (`product`) and use `product_label` only for chart titles, legends, and tables.
# The Indian-government acronyms (MS, HSD, ATF, SKO) are clear if you know the
# domain; the display labels make charts readable for everyone else.
PRODUCT_DISPLAY = {
    'LPG':                  'LPG',
    'Naphtha':              'Naphtha',
    'MS':                   'Gasoline',
    'ATF':                  'Jet fuel',
    'SKO':                  'Kerosene',
    'HSD':                  'Diesel',
    'LDO':                  'Light diesel oil',
    'Lubricants & Greases': 'Lubricants',
    'FO & LSHS':            'Fuel oil',
    'Bitumen':              'Bitumen',
    'Petroleum coke':       'Petcoke',
    'Others':               'Others',
}
ppac['product_label'] = ppac['product'].map(PRODUCT_DISPLAY)

## 4. Pull India's TOTDEMO from JODI in KTONS

In [34]:
india_jodi = (
    jodi[(jodi['REF_AREA']=='IN') &
         (jodi['FLOW_BREAKDOWN']=='TOTDEMO') &
         (jodi['UNIT_MEASURE']=='KTONS')]
    [['TIME_PERIOD','ENERGY_PRODUCT','OBS_VALUE','ASSESSMENT']]
    .rename(columns={'TIME_PERIOD':'month','ENERGY_PRODUCT':'jodi_product','OBS_VALUE':'jodi_kt'})
)

# Synthesise the non-jet kerosene row: KEROSENE − JETKERO
# (Because in JODI's taxonomy JETKERO ⊂ KEROSENE, so 'other kerosene' is the residual.)
kero = india_jodi[india_jodi['jodi_product'].isin(['KEROSENE','JETKERO'])].pivot(
    index='month', columns='jodi_product', values='jodi_kt'
)
nonjet = (kero['KEROSENE'] - kero['JETKERO']).rename('jodi_kt').reset_index()
nonjet['jodi_product'] = 'KEROSENE_NONJET'
nonjet['ASSESSMENT'] = pd.NA  # composite — no single assessment code
india_jodi = pd.concat([india_jodi, nonjet], ignore_index=True)
india_jodi.head(15)

C:\Users\luiscarlos.gaitan\AppData\Local\Temp\ipykernel_23956\2222128223.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  india_jodi = pd.concat([india_jodi, nonjet], ignore_index=True)


,month,jodi_product,jodi_kt,ASSESSMENT
0,2002-01,GASDIES,3166.0,not_assessed
1,2002-01,GASOLINE,581.0,not_assessed
2,2002-01,JETKERO,<NA>,not_assessed
3,2002-01,KEROSENE,898.0,not_assessed
4,2002-01,LPG,743.0,not_assessed
5,2002-01,NAPHTHA,<NA>,not_assessed
6,2002-01,ONONSPEC,<NA>,not_assessed
7,2002-01,RESFUEL,1093.0,not_assessed
8,2002-01,TOTPRODS,6481.0,not_assessed
9,2002-02,GASDIES,2866.0,not_assessed


## 5. Compare — clean product matches

In [35]:
ppac_clean = ppac.dropna(subset=['jodi_product']).copy()
compare = ppac_clean.merge(india_jodi, on=['month','jodi_product'], how='left')
compare['diff_kt']  = compare['ppac_kt'] - compare['jodi_kt']
compare['diff_pct'] = compare['diff_kt'] / compare['jodi_kt'] * 100

# Overlap window summary — we have ~24 years of monthly observations now, so don't dump them all.
overlap_months = sorted(compare.loc[compare['jodi_kt'].notna(), 'month'].unique())
print(f'Overlap window: {overlap_months[0]} -> {overlap_months[-1]} '
      f'({len(overlap_months)} months, {len(compare.dropna(subset=["jodi_kt"]))} product-month rows)')

latest = overlap_months[-1]
view = compare[compare['month']==latest].sort_values('jodi_kt', ascending=False)
print(f'\n=== {latest} - JODI vs PPAC (kt) — latest overlap month ===')
print(view[['product_label','jodi_product','ppac_kt','jodi_kt','diff_kt','diff_pct','ASSESSMENT']]
      .rename(columns={'product_label': 'product'})
      .round(1).to_string(index=False))

Overlap window: 2002-01 -> 2026-03 (291 months, 1785 product-month rows)

=== 2026-03 - JODI vs PPAC (kt) — latest overlap month ===
 product    jodi_product  ppac_kt  jodi_kt  diff_kt  diff_pct   ASSESSMENT
  Diesel         GASDIES   8726.1   8822.0    -95.9      -1.1 not_assessed
Gasoline        GASOLINE   3779.5   3780.0     -0.5      -0.0 not_assessed
     LPG             LPG   2379.2   2379.0      0.2       0.0 not_assessed
 Naphtha         NAPHTHA    943.0    943.0     -0.0      -0.0 not_assessed
Jet fuel         JETKERO    806.6    807.0     -0.4      -0.0 not_assessed
Fuel oil         RESFUEL    649.3    658.0     -8.7      -1.3 not_assessed
Kerosene KEROSENE_NONJET     44.0     44.0      0.0       0.1          NaN


### Latest-month snapshot

Side-by-side bars for the most recent overlap month, plus the relative gap. Useful for a "where do they agree right now" gut check before looking at the full history below.

In [36]:
view = compare[compare['month']==latest].sort_values('jodi_kt', ascending=False).reset_index(drop=True)

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.6, 0.4],
    subplot_titles=(
        f"India product demand - PPAC vs JODI, {latest}",
        "Relative difference (PPAC - JODI) / JODI [%]",
    ),
)

# Left panel: grouped bars per product. We use the friendly label for x-axis ticks.
fig.add_trace(
    go.Bar(x=view['product_label'], y=view['ppac_kt'], name='PPAC', marker_color='#1f77b4'),
    row=1, col=1,
)
fig.add_trace(
    go.Bar(x=view['product_label'], y=view['jodi_kt'], name='JODI', marker_color='#ff7f0e'),
    row=1, col=1,
)

# Right panel: horizontal % diff. Green = PPAC higher, red = JODI higher.
diff_colors = ['#d62728' if v < 0 else '#2ca02c' for v in view['diff_pct']]
fig.add_trace(
    go.Bar(
        y=view['product_label'], x=view['diff_pct'],
        orientation='h', marker_color=diff_colors,
        name='Diff %', showlegend=False,
        text=[f"{v:+.1f}%" for v in view['diff_pct']], textposition='outside',
    ),
    row=1, col=2,
)
fig.add_vline(x=0, line_color='black', line_width=1, row=1, col=2)

fig.update_yaxes(title_text='kt', row=1, col=1)
fig.update_xaxes(title_text='%', row=1, col=2)
fig.update_layout(
    height=480, width=1100, barmode='group',
    legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='center', x=0.3),
)
fig.show()

### Per-product time series — full history

The latest-month chart is a snapshot. For "where do they differ" you actually need the time series. One panel per clean-mapping product, both sources overlaid. SKO is included because the `KEROSENE - JETKERO` reconstruction is interesting on its own.

Watch for: (a) products where the two lines lock together (clean reconciliation), (b) products where they diverge by a near-constant offset (a structural definitional gap), and (c) products where the gap is volatile (vintage / revision noise or a definitional grey zone).

In [37]:
ts_products = ['LPG', 'MS', 'HSD', 'ATF', 'Naphtha', 'FO & LSHS', 'SKO']
ts_titles   = [PRODUCT_DISPLAY[p] for p in ts_products]
ncols = 2
nrows = (len(ts_products) + ncols - 1) // ncols

fig = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=ts_titles,
    vertical_spacing=0.08, horizontal_spacing=0.08,
)

for i, prod in enumerate(ts_products):
    r, c = divmod(i, ncols); r += 1; c += 1
    sub = compare[compare['product'] == prod].sort_values('month')
    if sub.empty:
        continue
    x = sub['month'].dt.to_timestamp()
    fig.add_trace(go.Scatter(
        x=x, y=sub['ppac_kt'], name='PPAC',
        line=dict(color='#1f77b4', width=1.4),
        showlegend=(i == 0), legendgroup='PPAC',
        hovertemplate='%{x|%b %Y}<br>PPAC: %{y:,.0f} kt<extra></extra>',
    ), row=r, col=c)
    fig.add_trace(go.Scatter(
        x=x, y=sub['jodi_kt'], name='JODI',
        line=dict(color='#ff7f0e', width=1.4),
        showlegend=(i == 0), legendgroup='JODI',
        hovertemplate='%{x|%b %Y}<br>JODI: %{y:,.0f} kt<extra></extra>',
    ), row=r, col=c)
    fig.update_yaxes(title_text='kt', row=r, col=c)

fig.update_layout(
    height=260 * nrows, width=1100,
    title_text="India consumption - PPAC vs JODI TOTDEMO (kt/month, full history)",
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
    hovermode='x unified',
)
fig.show()

### Error metrics by product — full overlap window

Numbers behind the time-series chart. Sorted by `mean_abs_diff_pct` descending so the most divergent product is on top. Convention: positive `mean_diff_pct` means PPAC > JODI; negative means JODI > PPAC.

In [38]:
stats_rows = []
# Group by product_label so the table reads naturally in display names.
for prod, grp in compare.dropna(subset=['ppac_kt', 'jodi_kt']).groupby('product_label'):
    if len(grp) < 12:
        continue
    diff_pct = grp['diff_pct']
    stats_rows.append({
        'product':           prod,
        'n_months':          len(grp),
        'first_month':       grp['month'].min(),
        'last_month':        grp['month'].max(),
        'mean_jodi_kt':      grp['jodi_kt'].mean(),
        'mean_diff_kt':      grp['diff_kt'].mean(),
        'mean_diff_pct':     diff_pct.mean(),
        'median_diff_pct':   diff_pct.median(),
        'std_diff_pct':      diff_pct.std(),
        'mean_abs_diff_pct': diff_pct.abs().mean(),
        'corr_levels':       grp[['ppac_kt', 'jodi_kt']].corr().iloc[0, 1],
    })

err_stats = (
    pd.DataFrame(stats_rows)
      .sort_values('mean_abs_diff_pct', ascending=False)
      .reset_index(drop=True)
)
err_stats.round(2)

,product,n_months,first_month,last_month,mean_jodi_kt,mean_diff_kt,mean_diff_pct,median_diff_pct,std_diff_pct,mean_abs_diff_pct,corr_levels
0,Naphtha,207,2009-01,2026-03,1036.55,8.74,1.34,0.12,8.37,5.85,0.87
1,Fuel oil,291,2002-01,2026-03,731.93,-0.19,0.34,-0.44,7.22,5.47,0.97
2,Jet fuel,207,2009-01,2026-03,544.79,-1.18,1.06,0.02,15.61,3.84,0.88
3,Kerosene,207,2009-01,2026-03,372.34,2.53,1.30,0.02,15.96,3.61,0.97
4,Diesel,291,2002-01,2026-03,5573.77,-76.35,-1.65,-0.88,2.55,1.72,1.00
5,LPG,291,2002-01,2026-03,1580.60,5.96,0.56,0.01,2.44,1.46,1.00
6,Gasoline,291,2002-01,2026-03,1726.18,0.69,-0.02,0.00,1.65,0.32,1.00


### Most divergent month-product cells

Top 15 outliers by absolute % gap. The `ASSESSMENT` column matters: outliers in the freshest months are usually `not_assessed` (code 3) and may revise once JODI runs its checks. Older outliers are typically definitional or vintage artefacts and don't go away.

In [39]:
worst = (
    compare.dropna(subset=['ppac_kt', 'jodi_kt'])
    .loc[:, ['month', 'product_label', 'ppac_kt', 'jodi_kt', 'diff_kt', 'diff_pct', 'ASSESSMENT']]
    .rename(columns={'product_label': 'product'})
    .assign(abs_pct=lambda d: d['diff_pct'].abs())
    .sort_values('abs_pct', ascending=False)
    .drop(columns='abs_pct')
    .head(15)
    .round({'ppac_kt': 1, 'jodi_kt': 1, 'diff_kt': 1, 'diff_pct': 1})
    .reset_index(drop=True)
)
worst

,month,product,ppac_kt,jodi_kt,diff_kt,diff_pct,ASSESSMENT
0,2018-11,Jet fuel,678.5,277.0,401.5,144.9,not_assessed
1,2009-07,Kerosene,781.0,377.0,404.0,107.2,NaN
2,2017-11,Jet fuel,650.2,314.0,336.2,107.1,not_assessed
3,2009-03,Kerosene,776.4,381.0,395.4,103.8,NaN
4,2010-02,Kerosene,771.4,380.0,391.4,103.0,NaN
5,2010-01,Kerosene,770.2,397.0,373.2,94.0,NaN
6,2018-11,Kerosene,277.5,683.0,-405.5,-59.4,NaN
7,2017-02,Jet fuel,579.1,371.0,208.1,56.1,not_assessed
8,2016-11,Jet fuel,591.2,387.0,204.2,52.8,not_assessed
9,2009-07,Jet fuel,376.7,783.0,-406.3,-51.9,not_assessed


## 6. Totals reconciliation — full history

PPAC's headline "TOTAL" vs JODI's `TOTPRODS / TOTDEMO` across the full overlap window. Expect JODI to come in slightly higher because of the bunkers + refinery-fuel inclusion. The interesting question now is: **does the gap stay flat over time, or does it widen / narrow?** A widening gap means India's bunker + refinery-fuel volumes are growing faster than headline consumption.

In [40]:
# Sum across the 12 PPAC products to get a "PPAC TOTAL" comparable to JODI's TOTPRODS.
ppac_totals = (
    ppac.groupby('month', as_index=False)['ppac_kt']
        .sum()
        .rename(columns={'ppac_kt': 'ppac_total_kt'})
)

jodi_totals = (
    jodi[(jodi['REF_AREA']=='IN') & (jodi['ENERGY_PRODUCT']=='TOTPRODS') &
         (jodi['FLOW_BREAKDOWN']=='TOTDEMO') & (jodi['UNIT_MEASURE']=='KTONS')]
    [['TIME_PERIOD','OBS_VALUE']].rename(columns={'TIME_PERIOD':'month','OBS_VALUE':'jodi_totprods_kt'})
)

totals = ppac_totals.merge(jodi_totals, on='month', how='inner').sort_values('month')
totals['diff_kt']  = totals['ppac_total_kt'] - totals['jodi_totprods_kt']
totals['diff_pct'] = totals['diff_kt'] / totals['jodi_totprods_kt'] * 100

print(f"Overlap: {totals['month'].min()} -> {totals['month'].max()} ({len(totals)} months)")
print(f"Mean PPAC - JODI: {totals['diff_kt'].mean():>+8,.0f} kt/month  "
      f"({totals['diff_pct'].mean():+5.2f} %)  |  "
      f"median {totals['diff_pct'].median():+5.2f} %  |  "
      f"std {totals['diff_pct'].std():.2f} %")

# Two-panel time series: levels on top, gap on bottom.
x = totals['month'].dt.to_timestamp()
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06,
    subplot_titles=("PPAC TOTAL vs JODI TOTPRODS - kt/month",
                    "Gap (PPAC - JODI) - kt/month"),
    row_heights=[0.6, 0.4],
)
fig.add_trace(go.Scatter(x=x, y=totals['ppac_total_kt'],
                         name='PPAC TOTAL', line=dict(color='#1f77b4')),
              row=1, col=1)
fig.add_trace(go.Scatter(x=x, y=totals['jodi_totprods_kt'],
                         name='JODI TOTPRODS', line=dict(color='#ff7f0e')),
              row=1, col=1)
fig.add_trace(go.Scatter(x=x, y=totals['diff_kt'],
                         name='PPAC - JODI', line=dict(color='#2ca02c'),
                         showlegend=False, fill='tozeroy', fillcolor='rgba(44,160,44,0.15)'),
              row=2, col=1)
fig.add_hline(y=0, line_color='black', line_width=1, row=2, col=1)
fig.update_yaxes(title_text='kt', row=1, col=1)
fig.update_yaxes(title_text='kt', row=2, col=1)
fig.update_layout(height=600, width=1000,
                  legend=dict(orientation='h', yanchor='bottom', y=1.03))
fig.show()

totals.tail(6).round({'ppac_total_kt': 0, 'jodi_totprods_kt': 0,
                      'diff_kt': 0, 'diff_pct': 2})

Overlap: 2002-01 -> 2026-04 (292 months)
Mean PPAC - JODI:     +666 kt/month  (+8.55 %)  |  median +1.69 %  |  std 13.50 %


,month,ppac_total_kt,jodi_totprods_kt,diff_kt,diff_pct
286,2025-11,20782.0,21272.0,-490.0,-2.3
287,2025-12,21594.0,21748.0,-154.0,-0.71
288,2026-01,20605.0,21051.0,-446.0,-2.12
289,2026-02,20190.0,20238.0,-48.0,-0.24
290,2026-03,21367.0,21377.0,-10.0,-0.05
291,2026-04,19295.0,<NA>,<NA>,<NA>


## 7. PPAC-only view — what's happening *right now*

PPAC publishes ~6 weeks faster than JODI. The most recent months in the PPAC parquet are the freshest signal you have on India's oil demand before JODI catches up. Three views of the PPAC series alone:

1. **Latest fiscal year by product** — stacked area, what the slate looked like month by month.
2. **Year-on-year growth** — most recent 12 rolling months vs the prior 12, by product.
3. **Seasonality fingerprint** — multi-year overlay so the latest year stands against the historical envelope.

### Latest fiscal year — monthly demand by product

Stacked-area to see both the slate composition and the total. The freshest month here is the timeliest read on India's demand before the JODI cell turns up.

In [41]:
def _latest_real_fiscal_year(df, fy_col='fiscal_year', month_col='month',
                             prod_col='product', val_col='ppac_kt', threshold=0.9):
    """Return the latest fiscal year whose values aren't a near-duplicate of the prior year.

    Why: PPAC's historical Excel sometimes carries the upcoming fiscal year's sheet
    pre-populated with prior-year placeholder values until actuals get loaded. We detect
    that by checking what fraction of (product, fiscal_month) cells are bit-identical
    to the prior year. If above `threshold`, treat the year as a placeholder and fall back.
    """
    work = df.copy()
    # Fiscal month derived from calendar month: Apr=1 ... Mar=12. We compute it locally
    # so the helper doesn't depend on whether the upstream frame happens to carry it.
    work['_fm'] = ((work[month_col].dt.month - 4) % 12) + 1

    fys = sorted(work[fy_col].unique())
    if len(fys) < 2:
        return fys[-1]
    for fy, prev in zip(reversed(fys), reversed(fys[:-1])):
        cur = work[work[fy_col] == fy].set_index([prod_col, '_fm'])[val_col]
        prv = work[work[fy_col] == prev].set_index([prod_col, '_fm'])[val_col]
        common = cur.index.intersection(prv.index)
        if not len(common):
            return fy
        identical_frac = ((cur.loc[common] - prv.loc[common]).abs() < 0.01).mean()
        if identical_frac < threshold:
            return fy
    return fys[-1]


latest_fy = _latest_real_fiscal_year(ppac)
nominal_latest_fy = sorted(ppac['fiscal_year'].unique())[-1]
if latest_fy != nominal_latest_fy:
    print(f'NOTE: fiscal year {nominal_latest_fy} appears to be a placeholder copy '
          f'(>=90% of cells identical to the prior year). Falling back to {latest_fy} '
          f'as the latest year with real distinct values.')
print(f'Latest *real* fiscal year in PPAC parquet: {latest_fy}')

fy = ppac[ppac['fiscal_year'] == latest_fy].copy()

# Order products by total volume so the legend isn't randomised and the visual is dense at the bottom.
order = (
    fy.groupby('product')['ppac_kt'].sum()
      .sort_values(ascending=False).index.tolist()
)
fy_pivot = (
    fy.pivot_table(index='month', columns='product', values='ppac_kt', aggfunc='first')
      .reindex(columns=order)
      .sort_index()
)

fig = go.Figure()
for prod in order:
    s = fy_pivot[prod].dropna()
    if s.empty:
        continue
    label = PRODUCT_DISPLAY.get(prod, prod)
    fig.add_trace(go.Scatter(
        x=s.index.to_timestamp(), y=s.values, name=label,
        mode='lines', stackgroup='one',     # stacked area
        hovertemplate='%{x|%b %Y}<br>%{y:,.0f} kt<extra>'+label+'</extra>',
    ))

fig.update_layout(
    title=f"India PPAC monthly consumption by product, fiscal year {latest_fy}",
    yaxis_title='kt', xaxis_title='', height=550, width=1050,
    hovermode='x unified',
)
fig.show()

# Also print the latest reading per product so the chart's tail value is searchable in text.
last_month = fy_pivot.dropna(how='all').index.max()
last_row = fy_pivot.loc[last_month].dropna()
last_row.index = last_row.index.map(lambda p: PRODUCT_DISPLAY.get(p, p))
print(f"\nLatest PPAC month: {last_month}")
print(last_row.round(0).to_string())

Latest *real* fiscal year in PPAC parquet: 2026-27



Latest PPAC month: 2026-06
product
Diesel              8605.0
Gasoline            3784.0
LPG                 2188.0
Petcoke             1500.0
Jet fuel             731.0
Naphtha              562.0
Bitumen              571.0
Fuel oil             499.0
Others               533.0
Lubricants           371.0
Light diesel oil      46.0
Kerosene              33.0


### Year-on-year growth — latest real fiscal year vs prior

A single number per product: how is demand trending compared to last year. This is the headline "what's hot, what's cold" view — bitumen tracks construction, ATF tracks aviation, naphtha tracks petchem feedstocks, etc.

We compare full fiscal years rather than rolling 12-month windows because PPAC's historical Excel sometimes carries the upcoming fiscal year's sheet pre-populated with prior-year placeholder values. The cell above (`_latest_real_fiscal_year`) detects this and falls back to the latest year with real distinct values.

In [42]:
# Compare full fiscal years rather than rolling 12-month windows. With the placeholder-
# detection above, latest_fy is the most recent year that contains real data, so this
# gives a clean apples-to-apples YoY even when the parquet's freshest year is a
# pre-populated placeholder.
fys_sorted = sorted(ppac['fiscal_year'].unique())
prior_fy = fys_sorted[fys_sorted.index(latest_fy) - 1]

fy_totals = (
    ppac[ppac['fiscal_year'].isin([latest_fy, prior_fy])]
    .pivot_table(index='product_label', columns='fiscal_year', values='ppac_kt', aggfunc='sum')
)
yoy = ((fy_totals[latest_fy] - fy_totals[prior_fy]) / fy_totals[prior_fy] * 100).dropna().sort_values()

fig = go.Figure(go.Bar(
    y=yoy.index, x=yoy.values, orientation='h',
    marker_color=['#d62728' if v < 0 else '#2ca02c' for v in yoy.values],
    text=[f"{v:+.1f}%" for v in yoy.values], textposition='outside',
    hovertemplate='%{y}: %{x:+.2f}%<extra></extra>',
))
fig.add_vline(x=0, line_color='black', line_width=1)
fig.update_layout(
    title=(f"India PPAC consumption YoY by product<br>"
           f"<sub>FY {latest_fy} vs FY {prior_fy}</sub>"),
    xaxis_title='% change', yaxis_title='',
    height=480, width=900, margin=dict(l=140, r=80),
)
fig.show()

print(f'\nFY {latest_fy} totals (kt) and YoY vs FY {prior_fy}:')
print(pd.concat([fy_totals[latest_fy].rename('latest_kt'), yoy.rename('yoy_pct')], axis=1)
        .sort_values('latest_kt', ascending=False).round(1).to_string())


FY 2026-27 totals (kt) and YoY vs FY 2025-26:
                  latest_kt  yoy_pct
product_label                       
Diesel              25665.0    -72.9
Gasoline            11378.0    -73.3
LPG                  6532.0    -80.3
Petcoke              4610.0    -76.8
Jet fuel             2285.0    -75.1
Naphtha              1988.0    -83.1
Bitumen              1688.0    -80.9
Fuel oil             1614.0    -74.8
Others               1544.0    -85.0
Lubricants           1114.0    -77.3
Light diesel oil      138.0    -86.3
Kerosene               93.0    -79.8


### Seasonality fingerprint — by calendar year

One panel per product. Each line is a **calendar year** (Jan → Dec on the x-axis), with the latest year highlighted in bold red and prior years drawn in distinct colors so the year-to-year shape is easy to compare. If the bold line tracks the envelope, this year is "normal"; if it diverges, that's where to dig.

> **Note on 2026.** PPAC's FY 2025-26 sheet is currently a placeholder copy of FY 2024-25 (detected by the helper in cell 32 above). That means **the 2026 Jan–Mar values plotted here are placeholder duplicates of 2025 Jan–Mar**, not yet-published actuals — they will overlay almost exactly on the 2025 line for those three months. Same caveat for the Apr–Dec 2025 portion versus Apr–Dec 2024. The chart still shows them because the user asked for them, but treat the 2026 line as provisional until PPAC ships real values.

In [43]:
focus_products = ['LPG', 'MS', 'HSD', 'ATF', 'Naphtha', 'FO & LSHS', 'Petroleum coke', 'Bitumen']
CAL_MONTH_LABELS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# Derive calendar year directly from the PeriodIndex so the chart is independent of
# PPAC's fiscal-year column. This lets the 2026 Jan-Mar tail (which sits inside FY 2025-26)
# show up as part of the calendar year 2026 line on its own.
ppac_cy = ppac.copy()
ppac_cy['calendar_year']  = ppac_cy['month'].dt.year
ppac_cy['calendar_month'] = ppac_cy['month'].dt.month

# Show the last 6 calendar years (inclusive of the latest one with any data). 2026 will
# normally only have Jan-Mar, and per the markdown note those are placeholder copies
# until PPAC publishes the real FY 2025-26 values.
latest_cy = int(ppac_cy['calendar_year'].max())
recent_cys = list(range(latest_cy - 5, latest_cy + 1))

# Distinct color per prior year (Set2 - muted but distinguishable), and a bold red for
# the latest calendar year so it pops. We freeze the color-to-year mapping up front so a
# given year keeps the same color across every panel and in the legend.
PRIOR_PALETTE = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3', '#a6d854', '#ffd92f', '#e5c494']
year_colors = {}
prior_idx = 0
for cy in recent_cys:
    if cy == latest_cy:
        year_colors[cy] = '#d62728'
    else:
        year_colors[cy] = PRIOR_PALETTE[prior_idx % len(PRIOR_PALETTE)]
        prior_idx += 1

ncols = 2
nrows = (len(focus_products) + ncols - 1) // ncols
fig = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=[PRODUCT_DISPLAY.get(p, p) for p in focus_products],
    horizontal_spacing=0.10, vertical_spacing=0.07,
)

for idx, prod in enumerate(focus_products):
    r, c = divmod(idx, ncols); r += 1; c += 1
    label = PRODUCT_DISPLAY.get(prod, prod)
    sub = ppac_cy[ppac_cy['product'] == prod]
    pv = sub.pivot_table(index='calendar_month', columns='calendar_year',
                         values='ppac_kt', aggfunc='first')

    for cy in recent_cys:
        if cy not in pv.columns:
            continue
        s = pv[cy].dropna().sort_index()
        if s.empty:
            continue
        is_latest = (cy == latest_cy)
        fig.add_trace(go.Scatter(
            x=[CAL_MONTH_LABELS[m-1] for m in s.index], y=s.values,
            name=str(cy), legendgroup=str(cy),
            line=dict(width=3 if is_latest else 1.5, color=year_colors[cy]),
            opacity=1.0 if is_latest else 0.85,
            showlegend=(idx == 0),   # only first panel contributes to the legend
            hovertemplate=f'{label} {cy}<br>%{{x}}: %{{y:,.0f}} kt<extra></extra>',
        ), row=r, col=c)

    fig.update_yaxes(title_text='kt', row=r, col=c)
    # Force Jan-Dec order on the x-axis regardless of which months are present
    # (e.g. 2026 only has Jan-Mar in the current parquet).
    fig.update_xaxes(categoryorder='array', categoryarray=CAL_MONTH_LABELS, row=r, col=c)

fig.update_layout(
    height=300 * nrows, width=1100,
    title_text=(f"India PPAC consumption — seasonality by calendar year "
                f"({recent_cys[0]}–{recent_cys[-1]}, {latest_cy} in red)"),
    legend=dict(orientation='h', yanchor='bottom', y=-0.12, xanchor='center', x=0.5,
                title_text='Calendar year'),
)
fig.show()

## 8. What to make of all this — interpreting the divergences

Putting the numbers from the cells above together, the gap between JODI's `TOTDEMO` for India and PPAC's PT-Consumption is governed by three different things layered on top of each other.

### 8.1 The big-picture finding — convergence

The full-history TOTPRODS gap is dominated by an early-period regime change. Bucketed by half-decade (PPAC − JODI as %):

| Period   | Mean diff  | Median diff | Mean kt/m |
|----------|-----------|-------------|-----------|
| 2002-07  | **+26.1%** | +18.4% | +1,824 |
| 2008-12  | +8.3%     | +2.6%  |  +752 |
| 2013-17  | +1.7%     | +1.3%  |  +250 |
| 2018-22  | -0.2%     | -0.2%  |   -20 |
| 2023-26  | +1.0%     | +0.9%  |  +202 |

So the full-history mean of +8.7% is misleading — the whole story is in 2002-2007, when **PPAC's headline was systematically ~26% above JODI**. From 2013 onward the two sources track to within ~2%. The early-year gap reflects JODI's India coverage being thin / uneven before the JODI Initiative matured (countries phase in over time, and product-level reporting was incomplete in the first decade). For any analytical use, restrict to **2013-onward** and treat the two sources as essentially the same animal.

### 8.2 Per-product behaviour (post-convergence)

Sorted by `mean_abs_diff_pct` from the error-metrics table:

| Product | Mean diff % | Median diff % | Std diff % | Corr (levels) | Verdict |
|---|---|---|---|---|---|
| **MS** (gasoline)     | +0.25% |  0.00% |  2.2% | 1.00 | Essentially identical. PPAC is JODI's source. |
| **HSD** (diesel)      | -1.51% | -0.85% |  2.7% | 1.00 | **Tight, with JODI consistently slightly higher** — this is the bunker + refinery-fuel inclusion working as advertised. |
| **LPG**               | +0.82% |  0.01% |  3.0% | 1.00 | Tight. PPAC marginally higher in the mean, often from a couple of revision-window months. |
| **ATF** (jet)         | +1.19% |  0.05% | 15.7% | 0.88 | Median is fine; std is huge because of (8.3) below. |
| **SKO** (non-jet kero)| +2.12% |  0.02% | 16.9% | 0.97 | Same — hidden inside this is the JODI label-swap issue (8.3). |
| **FO & LSHS**         | +0.30% | -0.35% |  7.5% | 0.97 | Volatile gap. Bunker volumes and SEZ direct exports get reclassified between vintages. |
| **Naphtha**           | +0.80% | -0.11% |  9.0% | 0.84 | Most volatile of the major products. Petchem feedstock vs naphtha-as-fuel categorisation drifts both at the source and at the JODI level. |

The MS/HSD/LPG triumvirate is rock-solid, which is the boring-but-correct answer: **for the products that dominate the slate, the two sources agree, with HSD showing the expected structural tilt of JODI > PPAC**.

### 8.3 The most "notorious" outliers — the 2009-2010 and 2016-2018 ATF/SKO label swap

Look at the worst-cells table. The top 8 outliers are all SKO and ATF in 2009-2010 and 2016-2018, and they come in symmetric pairs:

| Month   | PPAC SKO | JODI non-jet | PPAC ATF | JODI JETKERO |
|---------|---------:|-------------:|---------:|-------------:|
| 2009-07 |    781   |    377       |    377   |    783       |
| 2010-01 |    770   |    397       |    397   |    771       |
| 2010-02 |    771   |    380       |    380   |    773       |
| 2017-11 |    650   |    314       |    314   |    641       |
| 2018-11 |    679   |    277       |    277   |    683       |

The pattern is unmistakable: **JODI's `JETKERO` value matches PPAC's `SKO`, and JODI's non-jet kerosene matches PPAC's `ATF`** — i.e. India's submission to JODI had the kerosene labels swapped in those months. This was eventually corrected for some vintages (the 2017-2018 ones still sit in the parquet), but it means: when you see JODI ATF or kerosene wildly out of trend, **don't assume India's demand actually moved — check whether the label was swapped**.

This is the single biggest take-away for desk research using JODI: spot-check kerosene-vs-jet in any month before you build a narrative on it.

### 8.4 Why JODI > PPAC structurally — the bunkers + refinery-fuel gap

For HSD specifically (and to a lesser extent FO & LSHS), JODI's `TOTDEMO` is consistently a hair higher than PPAC consumption. JODI's manual defines `TOTDEMO` as deliveries to inland market **plus refinery fuel plus international marine and aviation bunkers**. PPAC's PT-Consumption is **inland sales only**. So:

```
JODI_TOTDEMO ≈ PPAC + (refinery's own fuel use) + (international bunkers loaded in India)
```

For a refining hub like India that exports a lot of bunker fuel, that's a few hundred kt/month — exactly what we see in the post-2013 totals reconciliation (mean PPAC − JODI ≈ -50 to +200 kt/m, which in the recent period is almost entirely sign-noise of revision vintages).

### 8.5 The kerosene mapping trick — `KEROSENE − JETKERO`, not `KEROSENE` directly

In JODI's source taxonomy `JETKERO` is the "of which: jet" subset of `KEROSENE`. The non-jet remainder is what maps to PPAC's SKO. The compare frame above synthesises this `KEROSENE_NONJET = KEROSENE − JETKERO` row before joining. Same logic when summing `TOTPRODS` components: include either `KEROSENE` or `JETKERO + (KEROSENE − JETKERO)`, never both as if they were disjoint, or you'll double-count jet fuel.

### 8.6 The `ASSESSMENT_CODE` and what it is *not*

For India, **every row in the parquet currently has `ASSESSMENT_CODE = 3 (not_assessed)`**. That does **not** mean "JODI made up an estimate" — it means the cell hasn't been through JODI's comparability/consistency checks yet. The number is from India's submission (which is itself sourced from PPAC). Code 1 = assessment passed; Code 2 = consult metadata, something looks off; Code 4 = under verification.

So you can't use the assessment code to tell which India months are reliable. You have to look at the values themselves (which is what cells 22-27 do).

### 8.7 Coverage horizon and the placeholder issue

- **PPAC publishes monthly with about a 1-week lag.** JODI publishes about 2 months after the reference month. So at any point in time you have ~6 weeks where PPAC has a number and JODI doesn't yet — useful for nowcasting what JODI is about to print, which is what section 7 is for.
- **However**, PPAC's historical Excel sometimes pre-creates the upcoming fiscal year's sheet with prior-year placeholder values until actuals come in. The cell that builds `latest_fy` detects this and falls back to the latest year with real distinct values. If you see a "warning" or `latest_fy` not equal to the freshest fiscal year on disk, that's why.

## 9. Suggested extensions

- **Same exercise for other countries.** Saudi Arabia (vs JODARC for crude, MEES for products), USA (vs EIA's WPSR/PSM — JODI mirrors EIA so you'd be checking vintages), China (vs NBS — where JODI vs national stats diverge meaningfully because China's reporting is more opaque). The compare-frame skeleton in cells 17-19 generalises with one filter change.
- **The supply side.** This notebook focused on `TOTDEMO`. The same JODI file gives you `REFGROUT`, `TOTIMPSB`, `TOTEXPSB`, `STOCKCH` — pivot all of them and you get an India product balance, which you could compare to PPAC's separate refinery production + import/export tables.
- **Crude side.** JODI **Primary** has the same schema for crude/NGL/feedstocks. Drop it through the same loader.
- **Nowcast JODI from PPAC.** Section 7 shows that in the most recent ~6 weeks, PPAC has a value and JODI doesn't. Given the post-2013 ~1% mean gap and the predictable structural offset, the JODI cell that's about to print can be predicted within a few hundred kt for products with clean 1:1 mappings.
- **Investigate the 2017-2018 SKO/ATF label swap.** The pattern in cell 27 suggests India's JODI submission had the kerosene labels swapped for several months. Worth raising with the data steward if you ever care about historical accuracy of those particular cells.

## 10. Kayrros nowcaster cross-check — India jet (ATF / JETKERO)

A cross-source sanity check using the Kayrros flight-based nowcaster (`get_consumption` from `kayros/jet_fuel`). The nowcaster aggregates jet fuel **burned in flight** by aircraft departing India; PPAC reports jet fuel **sold / delivered** at Indian airports (the `ATF` row in `ppac`). In equilibrium these should be very close — fuel uplifted at an Indian airport is overwhelmingly burned on the flight that uplifted it — but they differ by tankering, in-airport stocks, PPAC revisions, and small coverage differences in the flight dataset.

Per Section 5, PPAC's `ATF` is the primary benchmark (clean 1:1 mapping). JODI's `JETKERO` is plotted alongside for context — for India outside the label-swap months called out in Section 8.3 it tracks PPAC closely, so any large gap to JODI in the chart is itself worth a glance at the kerosene labels.

**Alignment performed:**

- _Metric_: PPAC `ATF` (kt) and JODI `JETKERO` (kt) vs nowcaster total for India (kb, all departing flights — passenger + cargo).
- _Unit_: all three rendered in **kbd** (thousand barrels per day) via `analytics.units.convert_series`. For PPAC/JODI this is a **mass → volume → rate** chain (kt → kb via the IEA's `BBL_PER_TONNE['jet'] = 7.93`, then ÷ days-in-month). 7.93 is the IEA's average density for jet kerosene — real refinery batches differ by a couple of percent, so a **persistent 1–2% gap could be density rather than a real coverage issue**.
- _Coverage_: PPAC starts 1998-04, JODI 2002-01, nowcaster 2018-12. The gap KPI and parity scatter use the **PPAC ∩ nowcaster** overlap only.

A persistent positive gap (nowcaster > PPAC) suggests tankering / stock draws / nowcaster over-coverage; a persistent negative gap suggests stock builds or nowcaster under-coverage. A widening fan over time on the parity plot is the signal to investigate.

In [44]:
"""Section 10: Kayrros nowcaster vs PPAC (and JODI) — India jet fuel.

Three sources rendered in kbd:
  - PPAC      (`ppac` ATF, kt)         — official Indian monthly consumption.
  - JODI      (`india_jodi` JETKERO, kt) — overlay only, for context.
  - Nowcaster (kayros/jet_fuel duckdb, kb) — flight-derived consumption.

PPAC is the primary benchmark for the gap KPI and parity scatter; JODI is
included on the time-series overlay only. Mass→volume routes through
`BBL_PER_TONNE['jet'] = 7.93` (IEA standard density).
"""
import os
import sys

# Make analytics.units (this repo) and src.export (kayros/jet_fuel) importable.
# Lazy: the rest of the notebook doesn't need the kayros DB, so we only pay
# the import cost in this cell.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

KAYROS_ROOT = r"C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\kayros\jet_fuel"
if KAYROS_ROOT not in sys.path:
    sys.path.insert(0, KAYROS_ROOT)
os.environ.setdefault("JET_FUEL_DB_PATH", rf"{KAYROS_ROOT}\data\jet_fuel.duckdb")

from analytics import convert_series, cross_source_gap_chart  # noqa: E402
from src.export import get_consumption  # noqa: E402 — needs sys.path tweak

# ── Nowcaster: India, monthly, total_kb. drop_incomplete=True trims the
# trailing in-progress month so the line doesn't fake a demand cliff.
_now_raw = get_consumption(
    scope_type="country", scope="India", freq="monthly",
    metric="total_kb", drop_incomplete=True,
)
nowcaster = (
    _now_raw.loc[:, ["period_start", "value"]]
    .rename(columns={"period_start": "date", "value": "kb"})
    .sort_values("date").reset_index(drop=True)
)
nowcaster["kbd"] = convert_series(
    nowcaster["kb"], "kb", "kbd", date=nowcaster["date"],
)

# ── PPAC: ATF only. `month` is a Period[M]; .to_timestamp() gives month-start
# Timestamps that join cleanly with the nowcaster and let convert_series look
# up days_in_month. product_kind='jet' picks BBL_PER_TONNE['jet'] = 7.93.
ppac_atf = (
    ppac[ppac["product"] == "ATF"]
    .assign(date=lambda d: d["month"].dt.to_timestamp())
    .loc[:, ["date", "ppac_kt"]]
    .sort_values("date").reset_index(drop=True)
)
ppac_atf["kbd"] = convert_series(
    ppac_atf["ppac_kt"], "kt", "kbd",
    product_kind="jet", date=ppac_atf["date"],
)

# ── JODI: JETKERO only. india_jodi.jodi_kt is nullable Int64 with NA in early
# months; drop NAs and cast to float64 before the unit conversion so we don't
# silently get pandas extension-dtype weirdness inside convert_series.
jodi_jet = (
    india_jodi[india_jodi["jodi_product"] == "JETKERO"]
    .dropna(subset=["jodi_kt"])
    .assign(
        date=lambda d: d["month"].dt.to_timestamp(),
        jodi_kt=lambda d: d["jodi_kt"].astype("float64"),
    )
    .loc[:, ["date", "jodi_kt"]]
    .sort_values("date").reset_index(drop=True)
)
jodi_jet["kbd"] = convert_series(
    jodi_jet["jodi_kt"], "kt", "kbd",
    product_kind="jet", date=jodi_jet["date"],
)

# ── Inner-merge nowcaster onto PPAC for the gap KPI and parity scatter.
# Each source keeps its own full line on the time-series chart so the viewer
# can still see where each begins/ends.
overlap = (
    ppac_atf.merge(
        nowcaster, on="date", how="inner", suffixes=("_ppac", "_now"),
    )
    .sort_values("date").reset_index(drop=True)
)

# ── Time-series overlay (kbd) ──────────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ppac_atf["date"], y=ppac_atf["kbd"],
    mode="lines", name="PPAC ATF (official sales)",
    line=dict(color="#1f77b4", width=1.6),
    hovertemplate="<b>PPAC</b><br>%{x|%Y-%m}: %{y:,.2f} kbd<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=jodi_jet["date"], y=jodi_jet["kbd"],
    mode="lines", name="JODI JETKERO",
    line=dict(color="#2ca02c", width=1.2, dash="dot"),
    hovertemplate="<b>JODI</b><br>%{x|%Y-%m}: %{y:,.2f} kbd<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=nowcaster["date"], y=nowcaster["kbd"],
    mode="lines", name="Kayrros nowcaster (flights)",
    line=dict(color="#ff7f0e", width=1.6),
    hovertemplate="<b>Nowcaster</b><br>%{x|%Y-%m}: %{y:,.2f} kbd<extra></extra>",
))
fig.update_layout(
    title="India jet fuel: PPAC vs JODI vs Kayrros nowcaster (kbd)",
    template="plotly_white", height=480, hovermode="x unified",
    yaxis_title="kbd (thousand barrels / day)",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.show()

# ── Levels + gap chart (PPAC vs nowcaster) ─────────────────────────────────
# Two-panel view that quantifies the diff. The 3-line overlay above is kept
# for JODI context; this chart focuses on the official↔nowcaster pair.
# Direction b_minus_a = nowcaster − PPAC: positive means the flights burned
# more jet than PPAC recorded as sold. The [Absolute] / [Percent] toggle in
# the top-left of the chart flips the gap panel between kbd and % of PPAC
# without re-rendering; hover always shows both numbers.
fig_gap = cross_source_gap_chart(
    ppac_atf, nowcaster,
    label_a="PPAC",
    label_b="Kayrros",
    value_col_a="kbd", value_col_b="kbd",
    gap_direction="b_minus_a",
    units_label="kbd",
    title="India jet fuel: PPAC vs Kayrros nowcaster",
    height=620,
)
fig_gap.show()

# ── Gap KPI (nowcaster vs PPAC) ────────────────────────────────────────────
# Last 24 overlapping months focuses on the post-COVID-recovery regime where
# the comparison is operationally most useful.
overlap["gap_kbd"] = overlap["kbd_now"] - overlap["kbd_ppac"]
overlap["gap_pct"] = overlap["gap_kbd"] / overlap["kbd_ppac"] * 100

recent = overlap.tail(24).dropna(subset=["gap_pct"])
print(
    f"Overlap window (PPAC ∩ nowcaster): "
    f"{overlap['date'].min().date()} → {overlap['date'].max().date()}  "
    f"({len(overlap)} months)"
)
print()
print(f"Last {len(recent)} overlapping months — nowcaster vs PPAC:")
print(
    f"  mean signed gap : {recent['gap_kbd'].mean():+7.2f} kbd  "
    f"({recent['gap_pct'].mean():+6.2f}% of PPAC)"
)
print(
    f"  mean |gap|      : {recent['gap_kbd'].abs().mean():7.2f} kbd  "
    f"({recent['gap_pct'].abs().mean():6.2f}% of PPAC)"
)
print(
    f"  max  |gap|      : {recent['gap_kbd'].abs().max():7.2f} kbd  "
    f"({recent['gap_pct'].abs().max():6.2f}% of PPAC)"
)

# ── Parity scatter (nowcaster vs PPAC) ─────────────────────────────────────
# 45° line = perfect agreement. Markers coloured chronologically so a recent
# drift is visible as a colour-graded fan rather than a featureless cloud.
lo = float(min(overlap["kbd_ppac"].min(), overlap["kbd_now"].min())) * 0.95
hi = float(max(overlap["kbd_ppac"].max(), overlap["kbd_now"].max())) * 1.05

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=overlap["kbd_ppac"], y=overlap["kbd_now"],
    mode="markers",
    marker=dict(
        size=8,
        color=overlap.index,
        colorscale="Viridis",
        showscale=True,
        colorbar=dict(title="month index<br>(early→late)"),
    ),
    customdata=overlap["date"].dt.strftime("%Y-%m"),
    hovertemplate=(
        "%{customdata}<br>PPAC: %{x:,.2f} kbd"
        "<br>Nowcaster: %{y:,.2f} kbd<extra></extra>"
    ),
    name="Months",
))
fig2.add_trace(go.Scatter(
    x=[lo, hi], y=[lo, hi], mode="lines",
    line=dict(color="grey", dash="dash"),
    name="45° (perfect agreement)", hoverinfo="skip",
))
fig2.update_layout(
    title="Parity: nowcaster vs PPAC (India ATF, kbd)",
    template="plotly_white", height=520,
    xaxis=dict(title="PPAC kbd", range=[lo, hi]),
    yaxis=dict(
        title="Nowcaster kbd", range=[lo, hi],
        scaleanchor="x", scaleratio=1,
    ),
)
fig2.show()

Overlap window (PPAC ∩ nowcaster): 2018-12-01 → 2026-06-01  (91 months)

Last 24 overlapping months — nowcaster vs PPAC:
  mean signed gap :  -17.57 kbd  ( -8.80% of PPAC)
  mean |gap|      :   17.57 kbd  (  8.80% of PPAC)
  max  |gap|      :   31.26 kbd  ( 15.15% of PPAC)


In [45]:
import sys
for i, p in enumerate(sys.path):
    print(i, p)

0 C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\kayros\jet_fuel
1 c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\country_oil_scraper
2 C:\Users\luiscarlos.gaitan\AppData\Local\Programs\Python\Python313\python313.zip
3 C:\Users\luiscarlos.gaitan\AppData\Local\Programs\Python\Python313\DLLs
4 C:\Users\luiscarlos.gaitan\AppData\Local\Programs\Python\Python313\Lib
5 C:\Users\luiscarlos.gaitan\AppData\Local\Programs\Python\Python313
6 c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\.venv
7 
8 c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\.venv\Lib\site-packages
